# Persistence & Checkpointing：InMemorySaver

> 适用版本：本项目锁定的 **LangGraph 1.1.2** 与 **langgraph-checkpoint 4.0.1**。本笔记不调用大模型或外部 API。

LangGraph 的 persistence（持久化）层会在图执行过程中保存 checkpoint（检查点）。一个 checkpoint 是某个时刻的 `StateSnapshot`：它包含当时的 State 值、下一批待执行节点、元数据，以及前一个 checkpoint 的引用。多个 checkpoint 通过 `thread_id` 组织成一条线程时间线。

| 概念 | 作用 | 是否属于业务 State |
| --- | --- | --- |
| State | 节点读取和更新的业务数据 | 是 |
| checkpoint | 某个 superstep 边界上的 State 快照与调度信息 | 否 |
| `thread_id` | 定位一条 checkpoint 时间线；相同 ID 恢复同一线程，不同 ID 相互隔离 | 否 |
| checkpointer | 负责 checkpoint 与节点级 pending writes 的读写 | 否 |

`thread_id` 放在 `config["configurable"]` 中，而不是 State 中：

```python
config = {"configurable": {"thread_id": "counter-demo"}}
```

本笔记将验证：

1. 首次 `invoke` 如何创建 checkpoint；
2. 第二次使用相同 `thread_id` 时，State 如何从上一次结果继续累积；
3. 不同 `thread_id` 为什么不会共享 State；
4. `InMemorySaver` 的生命周期、适用场景与限制。

```mermaid
flowchart LR
    I1[第一次 invoke: delta=5] --> C1[configurable.thread_id=memory-counter-demo]
    C1 --> L1[读取该线程最新 checkpoint: 首次为空]
    L1 --> A1[accumulate: total=5]
    A1 --> P1[保存 superstep checkpoint]
    P1 --> S1[summarize: 生成摘要]
    S1 --> P2[保存完成 checkpoint]
    P2 --> I2[第二次 invoke: delta=3 且 thread_id 相同]
    I2 --> L2[恢复 total=5 与历史记录]
    L2 --> A2[accumulate: total=8]
    A2 --> P3[追加新的 checkpoints]
```


In [1]:
import operator
from importlib.metadata import version
from typing import Annotated, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

print("LangGraph version:", version("langgraph"))
print("Checkpoint version:", version("langgraph-checkpoint"))


LangGraph version: 1.1.2
Checkpoint version: 4.0.1


## 1. 创建一个不依赖模型的状态图

下面使用 `accumulate → summarize` 两个顺序节点：

- 每次调用只输入本轮增量 `delta`；
- `accumulate` 读取 checkpoint 恢复出来的 `total`，再加上本轮 `delta`；
- `history` 使用 `operator.add` reducer，因此节点返回的新列表会追加到历史，而不是覆盖旧列表；
- `summarize` 根据最新总数生成文本摘要。

`TypedDict(total=False)` 让所有字段在类型层面都可省略。首次调用还没有 `total`，节点用 `state.get("total", 0)` 提供初始值；后续调用则从 checkpoint 中读到已有值。


In [2]:
class CounterState(TypedDict, total=False):
    delta: int
    total: int
    history: Annotated[list[str], operator.add]
    summary: str


def accumulate(state: CounterState) -> dict[str, int | list[str]]:
    """把本轮 delta 累加到 checkpoint 恢复出的 total。"""

    previous_total = state.get("total", 0)
    delta = state["delta"]
    new_total = previous_total + delta
    return {
        "total": new_total,
        "history": [f"{previous_total} + {delta} = {new_total}"],
    }


def summarize(state: CounterState) -> dict[str, str]:
    """根据已经提交的 total 生成本轮摘要。"""

    return {"summary": f"当前累计值：{state['total']}"}


def build_counter_graph(checkpointer):
    """构建同一张教学图；切换 saver 时只替换持久化后端。"""

    builder = StateGraph(state_schema=CounterState)
    builder.add_node("accumulate", accumulate)
    builder.add_node("summarize", summarize)
    builder.add_edge(START, "accumulate")
    builder.add_edge("accumulate", "summarize")
    builder.add_edge("summarize", END)
    return builder.compile(checkpointer=checkpointer)


memory_saver = InMemorySaver()
memory_graph = build_counter_graph(memory_saver)

print("图已使用 InMemorySaver 编译。")


图已使用 InMemorySaver 编译。


## 2. 第一次调用：创建线程与 checkpoint

`compile(checkpointer=...)` 只是为图装配 saver；真正运行时还必须提供 `thread_id`。这里先删除同名教学线程，让该单元格重复执行时仍从干净状态开始。`delete_thread()` 只清理这个明确指定的 ID，不影响其他线程。

本教程显式使用 `durability="sync"`：每个 superstep 的 checkpoint 会在下一步开始前同步写入 saver，便于紧接着检查历史。


In [3]:
MEMORY_THREAD_ID = "memory-counter-demo"
thread_config: RunnableConfig = {
    "configurable": {"thread_id": MEMORY_THREAD_ID}
}

# 仅清理本教程使用的确定性线程，保证从头运行 Notebook 时输出稳定。
memory_saver.delete_thread(MEMORY_THREAD_ID)

first_result = memory_graph.invoke(
    {"delta": 5},
    thread_config,
    durability="sync",
)

print("第一次调用结果：", first_result)

assert first_result["total"] == 5
assert first_result["history"] == ["0 + 5 = 5"]
assert first_result["summary"] == "当前累计值：5"


第一次调用结果： {'delta': 5, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}


### 结果解读

- 首次没有旧 checkpoint，因此 `total` 从节点提供的默认值 `0` 开始；
- 输入中的 `delta=5` 先写入线程 State，`accumulate` 再返回 `total=5` 和一条历史；
- `summarize` 在下一 superstep 读取已经提交的 `total=5`；
- 返回结果是当前最新 State，不是 checkpoint 底层序列化对象。


## 3. 检查 checkpoint 历史与保存时机

`get_state_history(config)` 返回**从新到旧**的 `StateSnapshot`。为了按执行顺序观察，下面先反转列表。对这张顺序图，LangGraph 1.1.2 的首次完整调用可看到四个边界：

| `metadata.step` | 代表的边界 | `next` |
| ---: | --- | --- |
| `-1` | 初始输入边界 | `__start__` |
| `0` | 本轮输入已写入 State | `accumulate` |
| `1` | `accumulate` 更新已提交 | `summarize` |
| `2` | `summarize` 更新已提交，图完成 | 空元组 |

> `step` 数字适合调试，不应作为业务主键。稳定定位某个历史快照应使用该快照 `configurable` 中的 `checkpoint_id`。


In [4]:
first_history_newest_first = list(
    memory_graph.get_state_history(thread_config)
)
first_history = list(reversed(first_history_newest_first))

for snapshot in first_history:
    print(
        f"step={snapshot.metadata['step']:>2} | "
        f"source={snapshot.metadata['source']:<5} | "
        f"next={snapshot.next} | values={dict(snapshot.values)}"
    )

checkpoint_ids = [
    snapshot.config["configurable"]["checkpoint_id"]
    for snapshot in first_history
]

assert [snapshot.metadata["step"] for snapshot in first_history] == [
    -1, 0, 1, 2
]
assert len(checkpoint_ids) == len(set(checkpoint_ids)) == 4
assert first_history[-1].next == ()
print("checkpoint 数量：", len(first_history))
print("每个 checkpoint 都有唯一 checkpoint_id：", True)


step=-1 | source=input | next=('__start__',) | values={'history': []}
step= 0 | source=loop  | next=('accumulate',) | values={'delta': 5, 'history': []}
step= 1 | source=loop  | next=('summarize',) | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5']}
step= 2 | source=loop  | next=() | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}
checkpoint 数量： 4
每个 checkpoint 都有唯一 checkpoint_id： True


## 4. 相同 `thread_id` 的第二次调用：恢复旧 State 后再执行

第二次只输入 `delta=3`，没有再次传入 `total` 或 `history`。LangGraph 会先按 `thread_id` 读取最新 checkpoint，将新输入合并到恢复出的 State，然后从 `START` 发起本轮新的图执行。

> 这里是“已完成线程上的新一轮调用”，不是从 `END` 节点继续执行。若图因 `interrupt()` 暂停，则应在相同 `thread_id` 上使用 `Command(resume=...)` 恢复中断点，那是另一种恢复语义。


In [5]:
first_latest_checkpoint_id = first_history_newest_first[0].config[
    "configurable"
]["checkpoint_id"]

second_result = memory_graph.invoke(
    {"delta": 3},
    thread_config,
    durability="sync",
)
latest_snapshot = memory_graph.get_state(thread_config)
all_history = list(memory_graph.get_state_history(thread_config))
second_latest_checkpoint_id = latest_snapshot.config["configurable"][
    "checkpoint_id"
]

print("第二次调用结果：", second_result)
print("最新 checkpoint 的 next：", latest_snapshot.next)
print("两轮调用后的 checkpoint 总数：", len(all_history))

assert second_result["total"] == 8
assert second_result["history"] == [
    "0 + 5 = 5",
    "5 + 3 = 8",
]
assert latest_snapshot.values == second_result
assert latest_snapshot.next == ()
assert second_latest_checkpoint_id != first_latest_checkpoint_id
assert len(all_history) == 8
# thread_id 属于运行配置，不会混入节点读取的业务 State。
assert "thread_id" not in latest_snapshot.values


第二次调用结果： {'delta': 3, 'total': 8, 'history': ['0 + 5 = 5', '5 + 3 = 8'], 'summary': '当前累计值：8'}
最新 checkpoint 的 next： ()
两轮调用后的 checkpoint 总数： 8


In [6]:
for snapshot in all_history:
    print(
        f"step={snapshot.metadata['step']:>2} | "
        f"source={snapshot.metadata['source']:<5} | "
        f"next={snapshot.next} | values={dict(snapshot.values)}"
    )

step= 6 | source=loop  | next=() | values={'delta': 3, 'total': 8, 'history': ['0 + 5 = 5', '5 + 3 = 8'], 'summary': '当前累计值：8'}
step= 5 | source=loop  | next=('summarize',) | values={'delta': 3, 'total': 8, 'history': ['0 + 5 = 5', '5 + 3 = 8'], 'summary': '当前累计值：5'}
step= 4 | source=loop  | next=('accumulate',) | values={'delta': 3, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}
step= 3 | source=input | next=('__start__',) | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}
step= 2 | source=loop  | next=() | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}
step= 1 | source=loop  | next=('summarize',) | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5']}
step= 0 | source=loop  | next=('accumulate',) | values={'delta': 5, 'history': []}
step=-1 | source=input | next=('__start__',) | values={'history': []}


### 结果解读

- `total` 从第一次结束时的 `5` 恢复，再加 `3` 得到 `8`；
- `history` 原有列表来自 checkpoint，本轮节点返回的列表通过 reducer 追加；
- 两轮调用各产生 4 个 checkpoint，因此本例历史共有 8 个；
- 最新快照的 `next=()` 表示当前线程已完成，没有待调度节点；
- 新 checkpoint 通过 `parent_config` 串到旧 checkpoint，构成可查询的时间线。


## 5. 线程隔离：更换 `thread_id` 就从独立状态开始

checkpointer 的索引不只包含 `thread_id`，底层还会使用 `checkpoint_ns` 与 `checkpoint_id`。根图的常规调用通常只需要显式提供 `thread_id`；`checkpoint_ns` 主要用于子图命名空间，`checkpoint_id` 用于定位历史中的特定快照。

业务上应选择稳定且不会碰撞的线程标识，例如会话 ID、工单 ID 或工作流实例 ID。不要把多个用户误用同一个固定 ID。


In [7]:
other_thread_config: RunnableConfig = {
    "configurable": {"thread_id": "memory-counter-other"}
}
memory_saver.delete_thread("memory-counter-other")

other_thread_result = memory_graph.invoke(
    {"delta": 2},
    other_thread_config,
    durability="sync",
)
original_thread_snapshot = memory_graph.get_state(thread_config)

print("新线程结果：", other_thread_result)
print("原线程仍保持：", dict(original_thread_snapshot.values))

assert other_thread_result["total"] == 2
assert original_thread_snapshot.values["total"] == 8
assert len(list(memory_graph.get_state_history(other_thread_config))) == 4


新线程结果： {'delta': 2, 'total': 2, 'history': ['0 + 2 = 2'], 'summary': '当前累计值：2'}
原线程仍保持： {'delta': 3, 'total': 8, 'history': ['0 + 5 = 5', '5 + 3 = 8'], 'summary': '当前累计值：8'}


## 6. 边界验证：缺少 `thread_id` 与 saver 生命周期

下面验证两个常见误区：

1. 图已经配置 checkpointer，但调用时没给可定位线程的配置；
2. 使用相同 `thread_id`，却换成了一个全新的 `InMemorySaver` 对象。

第二种情况下 ID 虽然相同，新 saver 的内存里却没有旧 checkpoint，因此仍会从空状态开始。


In [8]:
try:
    memory_graph.invoke({"delta": 1}, durability="sync")
except ValueError as exc:
    missing_thread_error = str(exc)
    print("缺少 thread_id 的预期错误：", type(exc).__name__)
    print(missing_thread_error)
else:
    raise AssertionError("预期缺少 thread_id 时抛出 ValueError")

assert "thread_id" in missing_thread_error

fresh_saver = InMemorySaver()
fresh_graph = build_counter_graph(fresh_saver)
fresh_result = fresh_graph.invoke(
    {"delta": 4},
    thread_config,  # ID 相同，但 saver 已经不是原对象。
    durability="sync",
)

print("全新 InMemorySaver 上的结果：", fresh_result)
assert fresh_result["total"] == 4
assert memory_graph.get_state(thread_config).values["total"] == 8


缺少 thread_id 的预期错误： ValueError
Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id
全新 InMemorySaver 上的结果： {'delta': 4, 'total': 4, 'history': ['0 + 4 = 4'], 'summary': '当前累计值：4'}


## 7. checkpoint 究竟在什么时候保存

Graph API 会在每个 **superstep 边界**生成完整 checkpoint。一个 superstep 中可能有多个并行节点；它们读取同一批已提交 State，全部完成后，更新才在边界合并为下一份完整快照。除此之外，节点任务完成时还可能写入 pending writes：如果同一 superstep 的另一个节点失败，恢复时已成功节点不必重新执行。

`invoke` / `stream` 的 `durability` 控制写入时序：

| 模式 | 物理保存时机 | 权衡 |
| --- | --- | --- |
| `"sync"` | 每步完成后同步持久化，再开始下一步 | 最强的逐步持久化保证，写入延迟在关键路径上 |
| `"async"`（默认） | 保存与下一步执行并行 | 吞吐更好，但进程突然退出时最近一步可能尚未落稳 |
| `"exit"` | 图退出时才保存 | 写入最少，但没有中间 checkpoint 的故障恢复能力 |

逻辑上的 checkpoint 边界与后端相同；`InMemorySaver` 和 `PostgresSaver` 的主要差异是数据放在哪里、能否跨进程/重启保留，以及运维与并发能力。


## 8. 适用场景、限制与最佳实践

### `InMemorySaver` 适合

- 单元测试、Notebook、概念验证和本地调试；
- 在同一个 Python 进程中演示多轮调用、interrupt、state history 或 time travel；
- 不希望引入数据库前置条件的最小示例。

### 主要限制

- 进程退出或 saver 对象被丢弃后，checkpoint 消失；
- 多进程/多副本之间不会自动共享数据；
- 数据量随线程与历史增长占用进程内存，不适合作为生产持久化后端；
- 它不是跨线程长期记忆。checkpointer 按 `thread_id` 保存线程 State；若要让多个线程共享用户资料，应使用 Store。

### 最佳实践

1. 把 `thread_id` 当作持久化主键管理，保证租户隔离且避免复用错误；
2. 只把可序列化、真正需要恢复的数据放入 State，不把数据库连接、文件句柄等运行时资源塞入 checkpoint；
3. 用 `get_state()` 检查最新快照，用 `get_state_history()` 检查时间线；不要依赖内部字典结构；
4. 生产环境根据延迟与恢复目标选择 durability，并使用 PostgreSQL 等持久后端；
5. 图或 State Schema 升级时，要考虑旧 checkpoint 与新代码的兼容性。

### 官方资料

- [LangGraph Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangGraph Checkpointing API Reference](https://reference.langchain.com/python/langgraph/checkpoints)
